In [ ]:
%pip install redis pandas

In [ ]:
import redis

In [ ]:
K_SYSMON = "winlogbeat"
K_SURICATA = "suricata"
K_SURICATA_ENRICHED = "pikksilm"

In [ ]:
R_INGEST = redis.Redis(host="127.0.0.1", port=6379, db=0)

In [ ]:
R_INGEST.delete(K_SYSMON)
R_INGEST.delete(K_SURICATA)
R_INGEST.delete(K_SURICATA_ENRICHED)

In [ ]:
count = 0
with open("../../test/winlog.json", "r") as handle:
    for line in handle:
        R_INGEST.rpush(K_SYSMON, line)
        count += 1
count

In [ ]:
R_INGEST.llen(K_SYSMON)

In [ ]:
count = 0
with open("../../test/suricata.json", "r") as handle:
    for line in handle:
        R_INGEST.rpush(K_SURICATA, line)
        count += 1
count

In [ ]:
import time

In [ ]:
time.sleep(10)

In [ ]:
R_INGEST.llen(K_SURICATA_ENRICHED)

In [ ]:
raw_messages = R_INGEST.lrange(K_SURICATA_ENRICHED, 0, -1)

In [ ]:
import json

In [ ]:
decoded = []
for msg in raw_messages:
    decoded.append(json.loads(msg))

In [ ]:
import pandas as pd

In [ ]:
DF = pd.json_normalize(decoded)

In [ ]:
# DF = DF.loc[DF["edr.process.entity_id"].notna()]

In [ ]:
COLS_CORE = ["event_type", "community_id", "src_ip", "dest_ip", "dest_port", "app_proto"]

In [ ]:
COLS_EDR = ["edr.process.name", "edr.process.parent.name", "edr.user.name", "edr.process.working_directory"]

In [ ]:
pd.set_option("display.max_colwidth", None)

In [ ]:
import ipywidgets as widgets

In [ ]:
def show(limit: int, columns: list, src_ip: list, dest_ip: list):
    pd.set_option('display.max_rows', limit)
    pd.set_option('display.min_rows', limit)
    df = (
        DF
        [list(columns)]
        .dropna(how="all", axis=1)
    )
    if len(src_ip) > 0:
        df = df.loc[df["src_ip"].isin(src_ip)]
    if len(dest_ip) > 0:
        df = df.loc[df["dest_ip"].isin(dest_ip)]
    return df

In [ ]:
widgets.interact(show, 
                 limit=widgets.IntSlider(min=10, max=1000), 
                 columns=widgets.SelectMultiple(options=list(DF.columns.values), 
                                                value=COLS_CORE + COLS_EDR, 
                                                rows=20),
                src_ip=widgets.SelectMultiple(options=sorted(DF["src_ip"].unique())),
                dest_ip=widgets.SelectMultiple(options=sorted(DF["dest_ip"].unique())))